# 04 — Registry Review & Evolution Governance

**Purpose**: Oversight of tool registry, lineage, and readiness for controlled evolution.

Evolution is currently OFF (`ENABLE_EVOLUTION = False`). This notebook prepares governance.

**Sections**:
1. Tool registry inventory (namespace + version)
2. Tool lineage from SQLite
3. Backtest delta score comparison
4. Correlation pruning report
5. Tools eligible for promotion
6. Tools eligible for deprecation

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from config import OUTPUTS_DIR, SQLITE_DB_FILE, ENABLE_EVOLUTION, GENERATED_TOOLS_DIR, TOOL_LIFECYCLE_FILE
from prediction_agent.storage.sqlite_store import SQLiteStore
from prediction_agent.evolution.tool_lifecycle_manager import ToolLifecycleManager

store = SQLiteStore()
lifecycle_mgr = ToolLifecycleManager()

print(f'ENABLE_EVOLUTION = {ENABLE_EVOLUTION}')
if ENABLE_EVOLUTION:
    print('⚠ WARNING: Evolution is ON — this notebook is intended for review mode (OFF).')
else:
    print('✓ Evolution is OFF. Safe to review registry.')

## 1. Tool Registry Inventory

In [ ]:
from tools.registry import build_default_registry
registry = build_default_registry()

tool_list = registry.list_tools()
generated = set(registry.generated_tool_names)

print(f'Total registered tools: {len(tool_list)}')
print(f'Generated tools: {len(generated)}')
print()

rows = []
for t in tool_list:
    name = t['name']
    rec = lifecycle_mgr.get_record(name)
    rows.append({
        'tool_name': name,
        'namespace': rec.namespace if rec else 'built-in',
        'version': rec.version if rec else 1,
        'is_generated': name in generated,
        'status': rec.status.value if rec else 'active',
        'usage_count': rec.usage_count if rec else 0,
        'correct_prediction_rate': round(rec.correct_prediction_rate, 3) if rec else None,
        'avg_latency_ms': round(rec.avg_latency_ms, 1) if rec else None,
        'description': t['description'][:60],
    })

registry_df = pd.DataFrame(rows)
print(registry_df[['tool_name','namespace','version','is_generated','status','usage_count']].to_string(index=False))

## 2. Tool Lineage from SQLite

In [ ]:
if SQLITE_DB_FILE.exists():
    lineage_rows = store.query('SELECT * FROM tools ORDER BY birth_timestamp')

    if lineage_rows:
        lineage_df = pd.DataFrame(lineage_rows)
        # Decode trigger_run_ids JSON
        lineage_df['trigger_run_ids'] = lineage_df['trigger_run_ids'].apply(
            lambda x: json.loads(x) if x else []
        )
        lineage_df['n_trigger_runs'] = lineage_df['trigger_run_ids'].apply(len)

        print('Tool Lineage in SQLite:')
        cols = ['tool_id', 'namespace', 'version', 'parent_tool_id', 'trigger_gap_id',
                'n_trigger_runs', 'capability_tag', 'status', 'backtest_delta_score']
        print(lineage_df[cols].to_string(index=False))

        # Tree visualization (text-based)
        print('\nLineage tree:')
        roots = lineage_df[lineage_df['parent_tool_id'].isna()]['tool_id'].tolist()
        children = lineage_df[lineage_df['parent_tool_id'].notna()].groupby('parent_tool_id')['tool_id'].apply(list)

        def _print_tree(tool_id, depth=0):
            prefix = '  ' * depth + ('└─ ' if depth > 0 else '')
            row = lineage_df[lineage_df['tool_id'] == tool_id]
            ns = row['namespace'].values[0] if len(row) else ''
            print(f'{prefix}{tool_id}  [{ns}]')
            for child in children.get(tool_id, []):
                _print_tree(child, depth+1)

        for root in roots:
            _print_tree(root)
        # Also show tools not in lineage_df (built-in, no SQLite entry)
        in_db = set(lineage_df['tool_id'].tolist())
        not_in_db = [t['name'] for t in tool_list if t['name'] not in in_db]
        if not_in_db:
            print('\nBuilt-in tools (no lineage entry yet):')
            for t in not_in_db:
                print(f'  • {t}  [built-in, not yet in SQLite tools table]')
    else:
        print('No lineage records in SQLite tools table yet.')
        print('Lineage is populated when tools are registered with register_tool_with_provenance().')
else:
    print('SQLite not found.')

## 3. Backtest Delta Score Comparison

In [ ]:
if SQLITE_DB_FILE.exists():
    delta_rows = store.query(
        'SELECT tool_id, namespace, backtest_delta_score, version FROM tools '
        'WHERE backtest_delta_score IS NOT NULL ORDER BY backtest_delta_score'
    )
    if delta_rows:
        delta_df = pd.DataFrame(delta_rows)
        fig, ax = plt.subplots(figsize=(10, 4))
        colors = ['green' if d < 0 else 'red' for d in delta_df['backtest_delta_score']]
        ax.barh(delta_df['tool_id'], delta_df['backtest_delta_score'], color=colors, alpha=0.8)
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title('Backtest Delta Score Per Tool\n(negative = improvement in Brier score)', fontsize=12)
        ax.set_xlabel('Δ Brier Score vs Baseline')
        good = mpatches.Patch(color='green', alpha=0.8, label='Improvement')
        bad = mpatches.Patch(color='red', alpha=0.8, label='Degradation')
        ax.legend(handles=[good, bad])
        plt.tight_layout()
        plt.show()
    else:
        print('No backtest_delta_score data yet.')
        print('Run backtests and update tool lifecycle records to populate this.')

## 4. Correlation Pruning Report

In [ ]:
SNAP_FILE = OUTPUTS_DIR / 'market_snapshots.jsonl'

if SNAP_FILE.exists():
    from prediction_agent.evolution.correlation_pruner import check_correlation, _pearson_r

    # Compute all tool pair correlations from tool_outputs in SQLite
    if SQLITE_DB_FILE.exists():
        pivot_rows = store.query("""
            SELECT run_id, tool_id, output_mean
            FROM tool_outputs
            ORDER BY run_id
        """)
        if pivot_rows:
            pivot_df = pd.DataFrame(pivot_rows).pivot_table(
                index='run_id', columns='tool_id', values='output_mean', aggfunc='mean'
            )
            corr = pivot_df.corr()
            tools = corr.columns.tolist()

            print('Tool Pair Pearson Correlations:')
            print(f'(|r| > 0.95 would be rejected by correlation pruner)')
            print()
            for i in range(len(tools)):
                for j in range(i+1, len(tools)):
                    r = corr.iloc[i, j]
                    flag = ' ⚠ WOULD REJECT' if abs(r) >= 0.95 else ''
                    print(f'  {tools[i]:<30s} ↔ {tools[j]:<30s}: r={r:+.4f}{flag}')
        else:
            print('No tool_outputs data in SQLite yet.')
    else:
        print('SQLite not found.')
else:
    print('No snapshot data available.')

## 5. Tools Eligible for Promotion (from pending/)

In [ ]:
pending_dir = GENERATED_TOOLS_DIR / 'pending'

pending_tools = list(pending_dir.glob('*.py')) if pending_dir.exists() else []
reviews = list(pending_dir.glob('*_REVIEW.md')) if pending_dir.exists() else []

print(f'Pending tools awaiting approval: {len(pending_tools)}')
print(f'Review documents: {len(reviews)}')

if pending_tools:
    print('\nTools in pending/:')
    for p in pending_tools:
        print(f'  • {p.name}')
        review = pending_dir / p.name.replace('.py', '_REVIEW.md')
        if review.exists():
            # Extract first few lines of review
            lines = review.read_text().splitlines()[:8]
            for l in lines:
                print(f'      {l}')
    print()
    print('To approve a tool, follow instructions in the _REVIEW.md file.')
else:
    print('No pending tools. Evolution is OFF — no new tools being generated.')
    print('Enable evolution manually when baseline data is sufficient.')

## 6. Tools Eligible for Deprecation

In [ ]:
from config import EVOLUTION_DEPRECATION_RUNS
from prediction_agent.evolution.schemas import ToolStatus

all_records = lifecycle_mgr.get_all_records()

print(f'Deprecation threshold: {EVOLUTION_DEPRECATION_RUNS} consecutive underperforming runs')
print()

deprecated = [r for r in all_records if r.status == ToolStatus.DEPRECATED]
at_risk = [r for r in all_records 
           if r.consecutive_underperformance > 0 
           and r.status != ToolStatus.DEPRECATED]

if deprecated:
    print(f'⚠ Deprecated tools ({len(deprecated)}):')
    for r in deprecated:
        print(f'  • {r.tool_name}  (consecutive_underperformance={r.consecutive_underperformance})')
else:
    print('✓ No deprecated tools.')

if at_risk:
    print(f'\nAt-risk tools ({len(at_risk)}) — consecutive underperformance > 0:')
    for r in at_risk:
        pct = r.consecutive_underperformance / EVOLUTION_DEPRECATION_RUNS * 100
        print(f'  • {r.tool_name}  ({r.consecutive_underperformance}/{EVOLUTION_DEPRECATION_RUNS} = {pct:.0f}% to deprecation)')
else:
    print('✓ No tools at risk of deprecation.')

if not all_records:
    print('No lifecycle records yet. Run the live loop to populate.')

# Summary table
if all_records:
    summary = []
    for r in all_records:
        summary.append({
            'tool': r.tool_name,
            'status': r.status.value,
            'usage': r.usage_count,
            'correct_rate': round(r.correct_prediction_rate, 3),
            'consec_underperf': r.consecutive_underperformance,
            'namespace': r.namespace,
        })
    print('\nFull lifecycle summary:')
    print(pd.DataFrame(summary).to_string(index=False))